# Faster R-CNN — Bayesian tuning, 8 trials (Kaggle T4 × 2)

Sẵn cấu hình tuning: sandbox / seed42 / cap100 / warmup3 / ES10 / shared test policy.
Tối đa8trial cho toàncampaign; mặc định mỗi Save & Run All chạy **một trial mới** với cả haiT4.
RunAll thực sự đăng ký và train, không chỉ preview. Không chạy song song hai agent.
Baseline: pilotES10 c5d2ptcg, validationmAP50–95=0.4276536057873146. Chọn bằngvalidation, khôngtest.
8trials là search; tốiđa4đốichứng sau winner chưa tự chạy trong notebook này.

Lần đầu: giữ CAMPAIGN_ACTION="new", không sửa setting nào khác. Add Input đúngcache,
InternetOn, T4x2 và SecretWANDB_API_KEY. Source training pin7746b79 đã kiểm cùngpilot.
Phiên kế tiếp: AddInput Output của phiên trước (hoặc uploadZIP/checksum), đổi
CAMPAIGN_ACTION="continue", điền RESTORE_BUNDLE đườngdẫnZIP nếu notebook không tìm được đúngmộtbundle.
Không đổi campaignname/source/search/policy để chạy tiếp. Khôi phụcSweep không resume trialdở.
Tải ZIP+SHA256 ở Outputgốc sau mỗi phiên, giữ mọi phiên. Không chạy lạiagent chỉ để export.

In [ ]:
import os, sys, subprocess, json, re
from pathlib import Path

REPO_URL = "https://github.com/tuan8p/VN-Traffic-Sign-Detection.git"
REPO_REF = "7746b79da7a86830ad4223030292ed09507fa905"  # SHA đầy đủ của commit có tích hợp Sweeps, đã push; không dùng tên branch.
assert re.fullmatch(r"[0-9a-f]{40}", REPO_REF), "Điền SHA đã commit/push trước khi chạy."
assert Path("/kaggle/working").exists(), "Notebook này dành cho Kaggle."
REPO = Path("/kaggle/working/VN-Traffic-Sign-Detection")

def run(command, **kwargs):
    return subprocess.run([str(x) for x in command], check=True, **kwargs)

if not (REPO / ".git").exists():
    run(["git", "clone", "--no-checkout", REPO_URL, REPO])
else:
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True)
    assert not dirty.strip(), "Repo có tracked changes; giữ lại trước khi checkout."
run(["git", "-C", REPO, "fetch", "origin", REPO_REF])
run(["git", "-C", REPO, "checkout", "--detach", REPO_REF])
os.chdir(REPO)
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[detect]", "wandb==0.31.0"])
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Source:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 1. Kiểm GPU và secret

Cell này chưa tạo run hoặc Sweep. API host lấy từ config chung của nhóm.
Lỗi CUDA/NMS hoặc thiếu GPU: dừng, không tự giảm batch hay số GPU.

In [ ]:
import torch, torchvision, wandb
from torchvision.ops import nms
from kaggle_secrets import UserSecretsClient
from vn_tsd.utils.io import load_yaml, save_yaml

shared = load_yaml(REPO / "configs/shared.yaml")
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
os.environ["WANDB_BASE_URL"] = shared["project"]["wandb_base_url"]
os.environ["WANDB_MODE"] = "online"
print("torch", torch.__version__, "torchvision", torchvision.__version__, "wandb", wandb.__version__)
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, "Cần đúng hai GPU T4."
for index in range(2):
    device = f"cuda:{index}"
    result = nms(torch.tensor([[0., 0., 10., 10.]], device=device), torch.tensor([.9], device=device), .5)
    assert result.tolist() == [0]
    print(index, torch.cuda.get_device_name(index))

## 2. Chuẩn bị/khôi phục cùng campaign — chưa train

Stage=tuning, Bayesian cap8, 100epochs/warmup3/ES10 đã điền sẵn; không có patience101.
SearchLR/WD/dropout/jitter lấy YAML đã pin. KhôngHyperband.
Tạo campaignmới đúngmột lần; phiênsau chọn continue, dùng bundle+checksum mới nhất đã lưu.
Nếu AddInput nhiều bundle cùngcampaign, điền RESTORE_BUNDLE trỏ đúngbundle mới nhất;
không tự đoán theo điểm số/tên. Continue khôngtìmthấybundle/ID thì dừng, khôngtạoSweepkhác.
Runtime/source/cachehash khác bị từchối; không sửa metadata để lách. Báo lỗi để xử lý.

In [ ]:
from vn_tsd.config.resolve import resolve_config
from vn_tsd.pipelines.faster_rcnn.sweep import prepare, load_campaign, PARAMETERS, OBJECTIVE
from vn_tsd.pipelines.faster_rcnn.dataset import content_hash
from zipfile import ZipFile
import hashlib, uuid, stat

STAGE = "tuning"
CAMPAIGN_NAME = "frcnn_tune_s42_es10_bayes8_sandbox_v1"
CAMPAIGN = Path("/kaggle/working/frcnn_sweeps") / CAMPAIGN_NAME
CAMPAIGN_ACTION = "new"  # lần đầu new; phiên sau continue để giữ cùng Sweep ID
RESTORE_BUNDLE = ""  # continue: để trống nếu AddInput có đúng1bundle campaign, hoặc điền path ZIP
DATA_ROOT = Path(load_yaml("configs/runtime/kaggle.yaml")["data_root_default"])
WANDB_PROJECT = "sandbox"
TEST_POLICY = "shared"
TUNING_RUN_CAP = 8
TUNING_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 10
WARMUP_EPOCHS = 3
BASELINE_RUN_ID = "c5d2ptcg"
BASELINE_VAL_MAP = 0.4276536057873146
assert CAMPAIGN_ACTION in {"new", "continue"}
assert DATA_ROOT.is_dir(), f"Kiểm AddInput cache: {DATA_ROOT}"

def restore_campaign(bundle_path, campaign):
    bundle_path, campaign = Path(bundle_path).resolve(), Path(campaign).resolve()
    checksum = bundle_path.with_suffix(bundle_path.suffix + ".sha256")
    assert checksum.is_file(), f"Thiếuchecksum: {checksum}"
    digest = hashlib.sha256()
    with bundle_path.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    assert digest.hexdigest() == checksum.read_text().split()[0], "Checksum khôngkhớp"
    assert not campaign.exists(), "Không ghi đè campaign đã có"
    campaign.parent.mkdir(parents=True, exist_ok=True)
    staging = campaign.parent / (".restore_" + uuid.uuid4().hex)
    staging.mkdir(exist_ok=False)
    with ZipFile(bundle_path) as archive:
        # Validate every destination before extraction; keep partial staging for diagnosis on error.
        for entry in archive.infolist():
            dest = (staging / entry.filename).resolve()
            assert "\\" not in entry.filename and dest.is_relative_to(staging / campaign.name), "Unsafe ZIP path"
            assert not stat.S_ISLNK(entry.external_attr >> 16), "Symlink ZIP entry rejected"
        assert archive.testzip() is None, "Bundle CRC error"
        archive.extractall(staging)
    restored = staging / campaign.name
    assert (restored / "sweep_id.json").is_file(), "Bundle thiếuSweepID; không đăng ký lại"
    assert restored.resolve().is_relative_to(campaign.parent), "Invalid restore location"
    restored.rename(campaign)
    print("Đãkhôi phục campaign:", campaign)

if not CAMPAIGN.exists() and CAMPAIGN_ACTION == "continue":
    if RESTORE_BUNDLE:
        bundle_path = Path(RESTORE_BUNDLE)
    else:
        candidates = sorted(Path("/kaggle/input").rglob(f"{CAMPAIGN_NAME}-artifacts-*.zip"))
        assert len(candidates) == 1, "Chọn đúng1bundle mới nhất +checksum ở Input hoặc điền RESTORE_BUNDLE"
        bundle_path = candidates[0]
    restore_campaign(bundle_path, CAMPAIGN)

overrides = {
    "data": {"root": str(DATA_ROOT)}, "seed": 42,
    "project": {"wandb_project": WANDB_PROJECT},
    "train": {"epochs": TUNING_EPOCHS, "early_stopping_patience": EARLY_STOPPING_PATIENCE,
              "warmup_epochs": WARMUP_EPOCHS},
}
cfg = resolve_config("configs/pipelines/faster_rcnn.yaml", "configs/shared.yaml",
                     "configs/runtime/kaggle.yaml", overrides=overrides)
search = load_yaml("configs/sweeps/faster_rcnn.yaml")
assert search["method"] == "bayes" and search["metric"] == {"name": OBJECTIVE, "goal": "maximize"}
assert "early_terminate" not in search
assert cfg["train"]["early_stopping_patience"] == 10 and cfg["train"]["epochs"] == 100
assert cfg["train"]["batch_size"] == 24 and cfg["eval"]["batch_size"] == 32
assert cfg["model"]["head"] == "custom_mlp" and cfg["online_preprocess"]["enabled"]
request = {"stage": STAGE, "cap": TUNING_RUN_CAP, "test_policy": TEST_POLICY, "cfg": cfg, "search": search}
if CAMPAIGN.exists():
    frozen, existing, existing_search = load_campaign(CAMPAIGN)
    assert json.loads((CAMPAIGN / "notebook_request.json").read_text()) == request, "Campaign settings changed"
    assert existing_search["run_cap"] == 8
    print("Giữ campaign đã chuẩn bị:", CAMPAIGN)
else:
    assert CAMPAIGN_ACTION == "new", "Continue không được tạo campaign mới"
    prepare(CAMPAIGN, cfg, search, run_cap=TUNING_RUN_CAP, nproc=2, test_policy=TEST_POLICY)
    (CAMPAIGN / "notebook_request.json").write_text(json.dumps(request, indent=2))
    (CAMPAIGN / "selection_policy.json").write_text(json.dumps({
        "baseline_run_id": BASELINE_RUN_ID, "baseline_val_map": BASELINE_VAL_MAP,
        "selection": "best validation mAP50-95 only; test not used",
        "search_attempt_cap": 8, "ablation_cap_after_review": 4,
        "failed_trials": "keep evidence; no automatic replacement outside approved cap",
    }, indent=2))
if CAMPAIGN_ACTION == "continue":
    assert (CAMPAIGN / "sweep_id.json").is_file(), "Continue thiếuID; không tạoSweep khác"
print((CAMPAIGN / "base_config.yaml").read_text())
print((CAMPAIGN / "sweep.yaml").read_text())

## 3. Đăng ký hoặc dùng lại Sweep — sandbox

REGISTER_SWEEP=True đã sẵn. Chỉlần đầu tạoSweep; cellregister dùng ID đã lưu khi chạy lại.
Nếu có registration_started.json mà thiếu ID, kiểmForge/khôi phụcID thật; không xóa marker và tạo lại mù.
Continue luôn yêu cầu ID đã có trước khi đến cellnày. Không đưa pilotnoES vào Sweep mới.

In [ ]:
REGISTER_SWEEP = True
if REGISTER_SWEEP:
    run([sys.executable, "-m", "vn_tsd.pipelines.faster_rcnn.sweep", "register", "--campaign", CAMPAIGN])
    print((CAMPAIGN / "sweep_id.json").read_text())
else:
    print("Chưa đăng ký. Bật REGISTER_SWEEP sau khi review config.")

## 4. Một trial mới mỗi phiên — tốiđa8 toàn Sweep

START_AGENT=True, AGENT_COUNT=1 đã sẵn. RunAll thực sự dùngGPU; mỗi trialtối đa khoảng4h+overhead.
Támtrial khôngphải támseed, mỗi trial là một bộLR/WD/dropout/jitter và khởi tạoCOCO mới.
Giới hạnlocal đếm cả failed/interrupted đãlưu; server run_cap8 cũng giữ nguyên khi tiếp tục cùngSweep.
Không sửa run_cap/campaignname để bùtriallỗi. Nếu quota/session mất thì trialdở khôngresume training.
Stop/PauseSweep để ngừng nhậntrialmới; đừng nhấn agenthai lần song song.
Khi đủ8, celldừng nhận thêm; chạyexport rồi gửi toàn bộartifacts đểreview bước4/chọnwinner.
Tốiđa4ablation chỉ chuẩn bị saureview, không tựchạy ởđây.

In [ ]:
START_AGENT = True
AGENT_COUNT = 1

def next_trial_count(campaign, requested, cap):
    assert type(requested) is int and requested > 0
    attempts = sum(path.is_dir() for path in (campaign / "trials").glob("*"))
    return min(requested, max(0, cap - attempts))

if START_AGENT:
    assert (CAMPAIGN / "sweep_id.json").is_file(), "CầnSweepID đã lưu"
    count = next_trial_count(CAMPAIGN, AGENT_COUNT, TUNING_RUN_CAP)
    if count:
        run([sys.executable, "-m", "vn_tsd.pipelines.faster_rcnn.sweep", "agent",
             "--campaign", CAMPAIGN, "--count", count])
    else:
        print("Đã đủ8trial attempts. Không bắt đầu thêm; chạyexport và review.")
else:
    print("Agent chưa chạy.")

## 5. Tải kết quả — một ZIP và checksum ở Output gốc

Cell này chỉ export, không đăng ký Sweep/train. Có thể chạy riêng sau cellagent lỗi hoặc training xong.
Bundle chứa metadata campaign và **mọi trial**: config, metrics/history, checkpoint, logs, figures;
không lồng run.zip hoặc checksum cũ để tránh nhân đôi checkpoint.
ZIP và .sha256 nằm trực tiếp trong /kaggle/working, tương tự notebook fulltrain trước đây.
Giữ cả failed/interrupted. SaveVersion giữ outputs; tải hai file đưa vào
`.agents/artifacts/notebooks/<campaign>/`. Forge hiện log metrics/figures, chưa tự upload checkpoint Artifact.

In [ ]:
import hashlib, uuid
from zipfile import ZipFile, ZIP_STORED
from IPython.display import display, FileLink

# Có thể copy nguyên cell này sang session cũ: CAMPAIGN phải trỏ campaign đã chạy.
assert CAMPAIGN.is_dir(), f"Chưa có campaign: {CAMPAIGN}"
bundle = Path("/kaggle/working") / f"{CAMPAIGN.name}-artifacts-{uuid.uuid4().hex[:8]}.zip"
files = sorted(path for path in CAMPAIGN.rglob("*")
               if path.is_file() and path.suffix.lower() not in {".zip", ".sha256"}
               and path.name not in {".netrc", "_netrc", ".env"})
assert files, "Campaign chưa có file để export"
with ZipFile(bundle, "x", compression=ZIP_STORED, allowZip64=True) as archive:
    for path in files:
        archive.write(path, f"{CAMPAIGN.name}/{path.relative_to(CAMPAIGN).as_posix()}")
with ZipFile(bundle) as archive:
    assert archive.testzip() is None, "Bundle CRC error"
digest = hashlib.sha256()
with bundle.open("rb") as stream:
    for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
        digest.update(block)
checksum = bundle.with_suffix(bundle.suffix + ".sha256")
checksum.write_text(f"{digest.hexdigest()}  {bundle.name}\n", encoding="utf-8")
print("Tải hai file ở Output /kaggle/working:")
print(bundle)
print(checksum)
display(FileLink(str(bundle)))
display(FileLink(str(checksum)))
for trial_dir in sorted((CAMPAIGN / "trials").glob("*")):
    status = trial_dir / "launch_status.json"
    print(trial_dir.name, json.loads(status.read_text()) if status.exists() else "chưa có launch receipt")

Nguồn SDK/command macros: [CoreWeave Sweeps walkthrough](https://docs.coreweave.com/products/wandb/sweeps/walkthrough),
[Sweep configuration](https://docs.coreweave.com/products/wandb/sweeps/sweep-config-keys).
Local mock/CPU-Gloo tests không thay nghiệm thu Kaggle CUDA/NCCL/Forge thật.